# D2a 합성곱과 풀링 — 실습 (W4)

> 위에서부터 한 셀씩 `Shift+Enter`로 실행하세요. `___` 빈칸은 직접 채웁니다.
> 이번 주는 **학습이 없습니다** — 부품(합성곱·풀링)의 원리를 손 계산·검산으로 완전히 익히는 주입니다.

**이 실습이 끝나면**
1. 설명서의 **손 계산(수직 경계 필터)** 을 `F.conv2d`로 재현한다
2. 진짜 사진에 필터 3종을 적용해 특징맵을 비교한다
3. 출력 크기 공식 **(in+2p−k)/s+1** 을 예측→확인한다
4. **풀링 그림의 숫자(9·5·6·8)** 를 재현하고, 이동 둔감성을 수치로 확인한다
5. Conv2d(448개) vs Linear(49,168개) 파라미터를 검산한다

**7단계 멘탈모델 초점:** 표현

## Part A. 데이터 준비 — CIFAR-10 미리 만나기
다음 주(D2b) 주인공인 컬러 사진 데이터셋. 오늘은 필터 실험용 이미지로만 씁니다.

In [ ]:
import torch                                            # PyTorch
import torch.nn as nn                                   # 신경망 모듈
import torch.nn.functional as F                         # 함수형 연산(conv2d 등)
import matplotlib.pyplot as plt                         # 그래프
from torchvision import datasets, transforms            # 영상 데이터

train_ds = datasets.CIFAR10('./data', train=True, download=True, transform=transforms.ToTensor())  # 컬러 32x32
classes = train_ds.classes                              # 클래스 10종
print('클래스:', classes)                               # airplane, automobile, ...
img, label = train_ds[7]                                # 이미지 한 장
print('shape:', img.shape, '| 클래스:', classes[label]) # (3,32,32) — 채널,높이,너비(D1a)

## Part B. 손 계산 재현 ⭐ — 수직 경계 필터
설명서 §4-1의 그림 그대로: 4×4 경계 이미지 + 2×2 필터 [[−1,1],[−1,1]] → 특징맵 가운데 열만 2.

In [ ]:
x4 = torch.tensor([[0., 0., 1., 1.],
                   [0., 0., 1., 1.],
                   [0., 0., 1., 1.],
                   [0., 0., 1., 1.]]).view(1, 1, 4, 4)  # (배치1, 채널1, 4, 4) — conv 입력 규격
filt = torch.tensor([[-1., 1.],
                     [-1., 1.]]).view(1, 1, 2, 2)       # 수직 경계 필터(손 설계)

feat = F.___(x4, filt)                                  # ✍️ 빈칸: 합성곱 함수 (F.?)
print('특징맵 (3x3):')                                  # 설명서 그림과 대조
print(feat[0, 0])                                       # 가운데 열만 2, 나머지 0
expected = torch.tensor([[0., 2., 0.]] * 3)             # 손 계산 예상값
print('손 계산과 일치?', bool(torch.equal(feat[0, 0], expected)))  # True면 검산 완료

## Part C. 진짜 사진에 필터 3종
수직 엣지·수평 엣지·윤곽(outline) 필터를 같은 사진에 적용해 특징맵을 비교합니다.
수평 필터는 수직 필터의 **행↔열을 뒤집으면**(전치) 됩니다.

In [ ]:
gray = img.mean(0, keepdim=True).unsqueeze(0)           # 흑백으로 (1,1,32,32) — D1a unsqueeze!
filt_v = torch.tensor([[-1., 0., 1.],
                       [-1., 0., 1.],
                       [-1., 0., 1.]]).view(1, 1, 3, 3) # 수직 엣지(좌우 밝기차)
filt_h = filt_v.transpose(2, ___)                        # ✍️ 빈칸: 마지막 두 차원(2,?)을 교환 → 수평 필터
filt_o = torch.tensor([[-1., -1., -1.],
                       [-1.,  8., -1.],
                       [-1., -1., -1.]]).view(1, 1, 3, 3)  # 윤곽(모든 방향 변화)

maps = [F.conv2d(gray, f, padding=1)[0, 0] for f in (filt_v, filt_h, filt_o)]  # 특징맵 3장
fig, axes = plt.subplots(1, 4, figsize=(12, 3))         # 원본+3장
axes[0].imshow(img.permute(1, 2, 0))                    # 원본(HWC로 변환)
axes[0].set_title(f'original: {classes[label]}')        # 제목(영어)
for ax, m, name in zip(axes[1:], maps, ['vertical edge', 'horizontal edge', 'outline']):
    ax.imshow(m, cmap='gray')                           # 특징맵
    ax.set_title(name)                                  # 제목(영어)
for ax in axes: ax.axis('off')                          # 축 끄기
plt.show()                                              # 필터마다 다른 것을 '본다'

> 같은 사진인데 필터마다 **다른 지도**가 나옵니다 — 수직 필터는 세로 경계에, 수평 필터는 가로 경계에 반응. CNN은 이런 필터 수십 개를 **학습으로** 찾습니다.

## Part D. 출력 크기 공식 — 예측 → 확인
**출력 = (입력 + 2p − k) / s + 1.** 실행 전에 각 shape을 먼저 계산해 보세요.

In [ ]:
x = torch.rand(1, 3, 32, 32)                            # 컬러 32x32 한 장
conv_same = nn.Conv2d(3, 16, 3, padding=___)            # ✍️ 빈칸: 크기 유지(same) 패딩
print('k3 p1 s1 →', conv_same(x).shape)                 # (1,16,32,32) — (32+2-3)/1+1=32
conv_valid = nn.Conv2d(3, 16, 3)                        # 패딩 없음
print('k3 p0 s1 →', conv_valid(x).shape)                # (1,16,30,30) — (32-3)/1+1=30
conv_half = nn.Conv2d(3, 16, 3, padding=1, stride=___)  # ✍️ 빈칸: 출력을 절반으로
print('k3 p1 s2 →', conv_half(x).shape)                 # (1,16,16,16) — (32+2-3)/2+1=16
print('채널: 3 → 16 (필터 16개 = 지도 16장)')            # 은닉 채널 = 특징맵 개수

## Part E. 풀링 — 그림 숫자 재현 + 이동 둔감성 ⭐
설명서 풀링 그림의 4×4를 그대로 입력해 [[9,5],[6,8]]이 나오는지 검산합니다.

In [ ]:
xp = torch.tensor([[1., 3., 2., 4.],
                   [2., 9., 5., 1.],
                   [6., 3., 8., 2.],
                   [1., 4., 0., 7.]]).view(1, 1, 4, 4)  # 그림의 입력 그대로
pooled = F.max_pool2d(xp, ___)                          # ✍️ 빈칸: 2x2 영역마다 최댓값
print('풀링 결과:')                                     # 그림과 대조
print(pooled[0, 0])                                     # [[9,5],[6,8]]

shifted = torch.roll(gray, shifts=1, dims=3)            # 사진을 오른쪽으로 1픽셀 이동
fm  = F.conv2d(gray,    filt_o, padding=1)              # 원본의 윤곽 특징맵
fm2 = F.conv2d(shifted, filt_o, padding=1)              # 이동본의 윤곽 특징맵
diff_raw    = (fm - fm2).abs().mean().item()            # 풀링 전 차이
diff_pooled = (F.max_pool2d(fm, 2) - F.max_pool2d(fm2, 2)).abs().mean().item()  # 풀링 후 차이
print('1픽셀 이동 시 특징맵 차이 — 풀링 전:', round(diff_raw, 4), '| 풀링 후:', round(diff_pooled, 4))
print('풀링 후 차이가 더 작다 =', diff_pooled < diff_raw, '→ 작은 이동에 둔감해짐')  # 효과 ③ 확인

## Part F. 파라미터 대결 — 공유의 위력 검산
같은 일(32×32×3 입력에서 특징 16종)을 Linear와 Conv2d로. D1c의 `numel` 검산법 그대로.

In [ ]:
conv = nn.Conv2d(3, 16, 3)                              # 3x3 필터 16개
n_conv = sum(p.___() for p in conv.parameters())        # ✍️ 빈칸: 원소 개수 메서드(D1c)
print('Conv2d(3,16,3)  :', n_conv, '개')                # 3*3*3*16+16 = 448

lin = nn.Linear(32 * 32 * ___, 16)                      # ✍️ 빈칸: 컬러 채널 수
n_lin = sum(p.numel() for p in lin.parameters())        # 같은 방식으로 검산
print('Linear(3072,16):', n_lin, '개')                  # 3072*16+16 = 49168
print('비율:', round(n_lin / n_conv), '배')             # ~110배 — 가중치 공유의 위력

## 🤖 AI 코파일럿 활용 (선택) — ai-native v1
막히면 AI 튜터에게 묻되, **먼저 스스로 생각**하고 답을 **실행으로 검증**하세요.

**좋은 질문 예시**
- "(32, k=5, p=2, s=1)의 출력 크기를 내가 공식으로 계산할 테니 채점해 줘."
- "대각선 경계를 찾는 3×3 필터를 내가 설계해 볼게 — 검산해 줘." (설계 후 Part C 코드로 직접 실험!)
- "풀링 전후의 차이 수치(Part E)가 왜 '이동 둔감성'의 증거인지 내 해석을 들어줘."
- "Conv2d(16, 32, 3)의 파라미터 수를 내가 계산할 테니 확인해 줘." (답: 3×3×16×32+32=4,640)

**가드레일**
1. 먼저 손으로 생각 → 그 다음 AI
2. AI 코드는 *왜 그런지* 설명할 수 있을 때만 사용
3. AI 출력은 실행으로 검증

## 정리 & 자가 점검

**오늘 한 일 3줄**
1. 손 계산(수직 경계→[0,2,0])을 `F.conv2d`로 재현했다 — 합성곱 = 미끄러지는 가중합
2. 출력 크기 공식과 풀링 숫자(9·5·6·8)를 검산하고, 이동 둔감성을 수치로 확인했다
3. Conv 448 vs Linear 49,168 — 가중치 공유가 파라미터를 110배 줄임을 검산했다

**스스로 점검**
- [ ] 필터가 '경계 위'에 있을 때만 2가 나오는 이유를 계산으로 보일 수 있다
- [ ] (in+2p−k)/s+1 공식으로 세 경우(32/30/16)를 재현할 수 있다
- [ ] 풀링 후 이동 차이가 작아지는 이유를 설명할 수 있다
- [ ] Conv2d(3,16,3)=448의 계산 과정(3×3×3×16+16)을 쓸 수 있다

**🔹심화 (선택)**
- **필터 설계 놀이:** 대각선(↘) 경계를 찾는 3×3 필터를 설계해 Part C에 넣어 보세요. 45° 회전한 경계 사진에도 반응하는지?
- **스트라이드 풀링:** `F.max_pool2d(xp, 2, stride=1)`로 바꾸면 출력이 어떻게 되는지 공식으로 예측 후 확인((4−2)/1+1=3×3).
- Part B의 필터를 [[1,−1],[1,−1]]로 뒤집으면 특징맵이 어떻게 변할지 예측→확인(부호 반전: 가운데 열 −2).